# 09 Drop and Alias

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Remove columns with <code>drop</code>, name columns and DataFrames with <code>alias</code>, and choose the right renaming strategy for one column, many columns, or a whole schema.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Clean output tables have clear names and no leftover helper columns. DataFrame aliases are essential for self-joins and for resolving ambiguous columns after joins, which is one of the most common errors people hit in PySpark.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Three related tools</b>
<ul><li><code>drop(*cols)</code> returns a new DataFrame without the listed columns. Missing names are <b>ignored</b></li><li><code>Column.alias("name")</code> names a column expression (also <code>.name()</code>, and <code>AS</code> in SQL)</li><li><code>DataFrame.alias("t")</code> names the whole DataFrame so you can write <code>F.col("t.id")</code>, which is needed for self-joins</li><li>Renaming options: <code>withColumnRenamed</code>, <code>withColumnsRenamed</code>, <code>select</code> with <code>alias</code>, <code>toDF(*new_names)</code></li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An HR system exports <code>EMP_NO</code>, <code>EMP_NM</code>, <code>DEPT_CD</code>, <code>SAL_AMT</code>, <code>SSN</code>, <code>TMP_FLAG</code>. The Silver table must rename everything to readable snake_case, drop the PII column <code>SSN</code> and the junk column <code>TMP_FLAG</code>, and join employees to their managers in the same table. That needs <code>drop</code>, a bulk rename, and a self-join with DataFrame aliases.
</div>

## Syntax

```python
df.drop("tmp_flag")                       # one column
df.drop("ssn", "tmp_flag")                # several
df.drop(*cols_to_drop)                    # from a list

F.col("sal_amt").alias("salary")          # column alias
df.alias("e")                             # DataFrame alias -> F.col("e.emp_no")

df.toDF("id", "name", "dept")             # rename all columns by position
df.withColumnsRenamed({"EMP_NO": "employee_id"})
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an HR export with cryptic upper-case names, a PII column and a junk column.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 5-row table with columns <code>EMP_NO</code>, <code>EMP_NM</code>, <code>DEPT_CD</code>, <code>SAL_AMT</code>, <code>MGR_NO</code>, <code>SSN</code>, <code>TMP_FLAG</code>.
</div>

In [0]:
from pyspark.sql import functions as F

hr_raw = spark.createDataFrame(
    [
        (1, "Amal", "D10", 21000, 3, "111-11-1111", "x"),
        (2, "Senthil", "D10", 150000, None, "222-22-2222", "x"),
        (3, "John", "D20", 48000, 2, "333-33-3333", None),
        (4, "Vivek", "D30", 25000, 3, "444-44-4444", "x"),
        (5, "Priya", "D20", 42000, 2, "555-55-5555", None),
    ],
    "EMP_NO INT, EMP_NM STRING, DEPT_CD STRING, SAL_AMT INT, MGR_NO INT, SSN STRING, TMP_FLAG STRING",
)
display(hr_raw)

## 1. Drop columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Drops one column, then several, then a list built in Python.<br><br>
<b>Why it matters</b><br>Dropping PII and helper columns before writing a table is a basic governance step.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
<code>drop</code> silently ignores names that don't exist. A typo means the column is <b>not</b> dropped and no error tells you. For sensitive columns, assert afterwards.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 columns after the first drop, 5 after the second, the same 5 from the list version, and then <code>True</code> from the PII assertion.
</div>

In [0]:
print(hr_raw.drop("TMP_FLAG").columns)
print(hr_raw.drop("TMP_FLAG", "SSN").columns)

to_drop = [c for c in hr_raw.columns if c in ("SSN", "TMP_FLAG")]
hr = hr_raw.drop(*to_drop)
print(hr.columns)

# Typo: nothing is dropped and no error is raised.
print(len(hr_raw.drop("SNN").columns), "columns after dropping a misspelled name")

# Defensive check for sensitive columns.
assert "SSN" not in hr.columns
print("PII removed:", "SSN" not in hr.columns)

## 2. Column aliases

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Gives readable names with <code>alias</code>, its synonym <code>name</code>, and SQL <code>AS</code>.<br><br>
<b>Why it matters</b><br>Every computed column needs a clear name. <code>alias</code> can also attach metadata, such as a comment, to a column.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three tables with the same readable column names: <code>employee_id</code>, <code>employee_name</code>, <code>annual_salary</code>.
</div>

In [0]:
hr.select(
    F.col("EMP_NO").alias("employee_id"),
    F.col("EMP_NM").name("employee_name"),        # .name() is the same as .alias()
    (F.col("SAL_AMT") * 1).alias("annual_salary"),
).show(2)

hr.selectExpr("EMP_NO AS employee_id", "EMP_NM AS employee_name", "SAL_AMT AS annual_salary").show(2)

# alias can attach metadata (for example a description) to the column.
with_meta = hr.select(F.col("SAL_AMT").alias("annual_salary", metadata={"comment": "Gross yearly salary in USD"}))
print(with_meta.schema["annual_salary"].metadata)

## 3. Renaming strategies compared

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Renames every column in four different ways.<br><br>
<b>Why it matters</b><br>Pick the right tool: <code>withColumnRenamed</code> for one or two, <code>withColumnsRenamed</code> for a mapping, <code>select</code> + <code>alias</code> for rule-based renames, and <code>toDF</code> when you know all new names in order.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Four DataFrames with the same final column names: <code>employee_id</code>, <code>employee_name</code>, <code>dept_code</code>, <code>salary</code>, <code>manager_id</code>.
</div>

In [0]:
mapping = {"EMP_NO": "employee_id", "EMP_NM": "employee_name", "DEPT_CD": "dept_code",
           "SAL_AMT": "salary", "MGR_NO": "manager_id"}

# a) mapping dict (Spark 3.4+)
r1 = hr.withColumnsRenamed(mapping)

# b) select + alias with the same mapping
r2 = hr.select([F.col(c).alias(mapping.get(c, c)) for c in hr.columns])

# c) toDF with all names in column order
r3 = hr.toDF(*[mapping[c] for c in hr.columns])

# d) rule-based: lower-case everything (no mapping needed)
r4 = hr.select([F.col(c).alias(c.lower()) for c in hr.columns])

print(r1.columns)
print(r1.columns == r2.columns == r3.columns)
print(r4.columns)

## 4. DataFrame aliases and self-joins

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins the employee table to itself to find each employee's manager, using DataFrame aliases <code>e</code> and <code>m</code>.<br><br>
<b>Why it matters</b><br>Without aliases, both sides have the same column names, so Spark can't tell which <code>employee_id</code> you mean and raises an ambiguity error. Aliases solve it cleanly. Joins are covered fully in lesson 17.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Each employee with their manager's name. Senthil has no manager (null), because of the left join.
</div>

In [0]:
emp = r1

e = emp.alias("e")
m = emp.alias("m")

(
    e.join(m, F.col("e.manager_id") == F.col("m.employee_id"), "left")
     .select(
         F.col("e.employee_name").alias("employee"),
         F.col("m.employee_name").alias("manager"),
     )
     .orderBy("employee")
     .show()
)

## 5. Edge case: dropping after a join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins two DataFrames that both have a <code>dept_code</code> column, then removes the duplicate using a DataFrame reference.<br><br>
<b>Why it matters</b><br>After a join on an expression, both key columns survive. <code>drop("dept_code")</code> with a string can be ambiguous. Dropping by the specific DataFrame's column, <code>drop(depts["dept_code"])</code>, removes the right one.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table with a single <code>dept_code</code> column plus <code>dept_name</code>.
</div>

In [0]:
depts = spark.createDataFrame(
    [("D10", "Data"), ("D20", "Finance"), ("D30", "Sales")], "dept_code STRING, dept_name STRING"
)

joined = emp.join(depts, emp["dept_code"] == depts["dept_code"], "inner")
print("Columns after join:", joined.columns)

clean = joined.drop(depts["dept_code"])
print("Columns after drop:", clean.columns)
clean.select("employee_name", "dept_code", "dept_name").show()

## Under the hood

```
df.drop("ssn")          -> Project [all columns except ssn]
col("x").alias("y")     -> Alias(x, y) inside a Project
df.alias("e")           -> SubqueryAlias e   (only a name for resolving columns)
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `drop`, `alias` and renames only change the plan's projection. No data is touched.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> no shuffle. Dropping a column also lets the optimizer skip reading it from columnar files.

`DataFrame.alias` adds a `SubqueryAlias` node. It doesn't copy data, it only gives the analyzer a qualifier to resolve `e.col` versus `m.col`.

In [0]:
spark.range(3).alias("t").select(F.col("t.id").alias("row_id")).drop("unused").explain(True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: column was not dropped</b><br>
<code>drop</code> ignores unknown names. Check spelling and case, then assert the column is gone.<br><br>
**⛔ Problem: <code>[AMBIGUOUS_REFERENCE]</code> after a join or self-join**<br>Both sides have the same column name. Use <code>df.alias("a")</code> and <code>F.col("a.col")</code>, or drop the duplicate via <code>drop(other_df["col"])</code>.<br><br>
**⛔ Problem: <code>toDF</code> fails with a size mismatch**<br><code>toDF</code> needs exactly one name per column, in order.<br><br>
<b>⛔ Problem: an alias is lost after a later transformation</b><br>Column aliases name the output of that <code>select</code> only. DataFrame aliases are carried forward, but re-alias after complex steps if references stop resolving.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What happens if you <code>drop</code> a column that doesn't exist?</b><br>
Nothing. It is a silent no-op, which is convenient but can hide typos, so critical drops should be asserted.<br><br>

<b>🎤 2. What is the difference between a column alias and a DataFrame alias?</b><br>
A column alias names an expression in the output. A DataFrame alias names the whole DataFrame so columns can be qualified, like <code>e.id</code> and <code>m.id</code>, which is required for self-joins.<br><br>

<b>🎤 3. How do you rename all columns of a DataFrame?</b><br>
Use <code>toDF(*new_names)</code> if I have the full list in order, <code>withColumnsRenamed(mapping)</code> for a dictionary, or <code>select([col(c).alias(f(c)) for c in df.columns])</code> for a rule such as lower-casing.<br><br>

<b>🎤 4. How do you remove the duplicate key column after a join?</b><br>
Join with <code>on="key"</code> (a string or list), which keeps one copy automatically, or drop the other side's column with <code>drop(other_df["key"])</code>.<br><br>

<b>🎤 5. Are <code>drop</code> and <code>alias</code> expensive?</b><br>
No. They only change the projection in the logical plan. Dropping columns can even make queries cheaper, because columnar readers skip them.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which statement about <code>df.drop("colX")</code> is true when <code>colX</code> doesn't exist in <code>df</code>?</b><br>
A. It raises an AnalysisException<br>
B. It returns <code>df</code> unchanged without an error<br>
C. It returns an empty DataFrame<br>
D. It drops the last column<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>drop</code> ignores column names that don't exist.</details><br><br>

<b>Q2. A self-join of <code>employees</code> with itself fails with an ambiguous reference error. What fixes it?</b><br>
A. Cache the DataFrame before joining<br>
B. Use <code>employees.alias("e")</code> and <code>employees.alias("m")</code> and qualify columns<br>
C. Use a cross join<br>
D. Repartition both sides by the join key<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Aliases give each side a qualifier so the analyzer can resolve columns.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>hr_raw</code>:
<ol><li>Drop <code>SSN</code> and <code>TMP_FLAG</code> using a list, and assert neither remains</li><li>Rename all columns to lower case with one <code>select</code></li><li>Rename using a dictionary to: <code>id</code>, <code>name</code>, <code>dept</code>, <code>salary</code>, <code>manager_id</code></li><li>Self-join to produce <code>employee</code>, <code>manager</code>, and <code>same_dept</code> (true if they share a department)</li><li>Join to the <code>depts</code> DataFrame and keep only one <code>dept</code> column</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Drop from a list and assert
drop_list = ["SSN", "TMP_FLAG"]
p = hr_raw.drop(*drop_list)
assert not set(drop_list) & set(p.columns)

# 2. Lower-case all names
p_lower = p.select([F.col(c).alias(c.lower()) for c in p.columns])
print(p_lower.columns)

# 3. Rename with a dict
p = p.withColumnsRenamed({"EMP_NO": "id", "EMP_NM": "name", "DEPT_CD": "dept", "SAL_AMT": "salary", "MGR_NO": "manager_id"})

# 4. Self-join with aliases
a, b = p.alias("a"), p.alias("b")
a.join(b, F.col("a.manager_id") == F.col("b.id"), "left").select(
    F.col("a.name").alias("employee"),
    F.col("b.name").alias("manager"),
    (F.col("a.dept") == F.col("b.dept")).alias("same_dept"),
).show()

# 5. Join on a column with a different name, then drop the duplicate
d = depts.withColumnRenamed("dept_code", "dept")
p.join(d, p["dept"] == d["dept"]).drop(d["dept"]).show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>drop(*cols)</code> removes columns and silently ignores missing ones</li><li><code>col.alias("x")</code> (or <code>.name("x")</code>, or SQL <code>AS</code>) names a column</li><li><code>df.alias("t")</code> names a DataFrame for qualified references and self-joins</li><li>Rename one: <code>withColumnRenamed</code>. Many: <code>withColumnsRenamed</code>, <code>select</code> + <code>alias</code>, or <code>toDF</code></li><li>After a join, drop the duplicate column via <code>drop(other_df["col"])</code></li><li>All of these only change the plan's projection and are cheap</li></ul>
</div>

| Task | Code |
|---|---|
| Drop | `df.drop("a", "b")`, `df.drop(*cols)` |
| Column alias | `F.col("a").alias("b")` |
| DataFrame alias | `df.alias("t")`, then `F.col("t.a")` |
| Rename all by position | `df.toDF("x", "y", "z")` |
| Rename by dict | `df.withColumnsRenamed({"a": "x"})` |
| Drop join duplicate | `joined.drop(right_df["key"])` |

## Git commit

```
git add 01_spark_basics/09_drop_alias.ipynb
git commit -m "add 09 drop alias notebook"
```